In [ ]:
import os

if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    import os

    os.environ["MODIN_ENGINE"] = "ray"
    import ray

    ray.init(
        num_cpus=int(os.environ["MODIN_CPUS"]),
        runtime_env={"env_vars": {"__MODIN_AUTOIMPORT_PANDAS__": "1"}},
    )
    import modin.pandas as pd
else:
    import pandas as pd

In [ ]:
### cell 0 ###

factor = 1
data = pd.read_csv(
    "/home/dias-benchmarks/notebooks/nickwan/creating-player-stats-using-tracking-data/input/nfl-big-data-bowl-2023/week1.csv"
)
scout = pd.read_csv(
    "/home/dias-benchmarks/notebooks/nickwan/creating-player-stats-using-tracking-data/input/nfl-big-data-bowl-2023/pffScoutingData.csv"
)
plays = pd.read_csv(
    "/home/dias-benchmarks/notebooks/nickwan/creating-player-stats-using-tracking-data/input/nfl-big-data-bowl-2023/plays.csv"
)
players = pd.read_csv(
    "/home/dias-benchmarks/notebooks/nickwan/creating-player-stats-using-tracking-data/input/nfl-big-data-bowl-2023/players.csv"
)

# Let's merge these data into one set
data = data.merge(scout, how="left")
data = data.sample(frac=factor, random_state=0)
data.shape

In [ ]:
### cell 1 ###

data.info()

In [ ]:
### cell 2 ###

# vectorized approach using groupby and cumcount
snap_group = data["event"].eq("ball_snap").cumsum()
offset = data.groupby(snap_group).cumcount()
# keep rows from each snap (offset 0) up to 5 rows after (offset <= 5)
df = data[(snap_group > 0) & (offset <= 5)]

In [ ]:
### cell 3 ###

gid = 2021090900
pid = 97
nid = 25511
df.loc[(df["gameId"] == gid) & (df["playId"] == pid) & (df["nflId"] == nid)]

In [ ]:
### cell 4 ###

# get line of scrimmage info to compute block/rush depth relative to LOS
_los = data.loc[
    (data["team"] == "football") & (data["frameId"] == 1), ["gameId", "playId", "x"]
].rename(columns={"x": "los"})

# merge LOS info back to subsetted data
df = df.merge(_los)

In [ ]:
### cell 5 ###

df.loc[(df["gameId"] == gid) & (df["playId"] == pid) & (df["nflId"] == nid)]

In [ ]:
### cell 6 ###

diff = df["x"] - df["los"]
df["los_diff"] = diff.where(df["playDirection"] != "left", -diff)

# Merge possession team info directly on the join keys
df = df.merge(plays[["gameId", "playId", "possessionTeam"]], on=["gameId", "playId"])

In [ ]:
### cell 7 ###

df.loc[(df["gameId"] == gid) & (df["playId"] == pid) & (df["nflId"] == nid)]

In [ ]:
### cell 8 ###

df["posTeam"] = (
    (df["possessionTeam"] == df["team"])
    .astype("int8")
    .where(df["team"] != "football", other=-1)
)

# create initial snap speed dataframe (mean of speed and acceleration per player)
snap_speed = df[["nflId", "s", "a"]].groupby("nflId", as_index=False).mean()

In [ ]:
### cell 9 ###

snap_speed.head()

In [ ]:
### cell 10 ###

# given whether a offense player or defense player, aggregate by maxmimum or minimum LOS difference, respectively.
# e.g. if o-lineman has more a negative LOS diff, they allow more pass rush penetration
off = (
    df.loc[df["posTeam"] == 1, ["gameId", "playId", "nflId", "los_diff"]]
    .groupby(["gameId", "playId", "nflId"], as_index=False)
    .max()
)
def1 = (
    df.loc[df["posTeam"] == 0, ["gameId", "playId", "nflId", "los_diff"]]
    .groupby(["gameId", "playId", "nflId"], as_index=False)
    .min()
)
los_diff = off._append(def1)
los_diff = (
    los_diff.loc[:, ["nflId", "los_diff"]].groupby("nflId", as_index=False).mean()
)

# merge LOS diff data back onto snap speed
snap_speed = snap_speed.merge(los_diff)
snap_speed = snap_speed.rename(
    columns={"s": "snap_s", "a": "snap_a", "los_diff": "snap_los_diff"}
)

In [ ]:
### cell 11 ###

snap_speed.head()

In [ ]:
### cell 12 ###

df_plt = players.loc[:, ["nflId", "officialPosition", "displayName"]].merge(snap_speed)

In [ ]:
### cell 13 ###

df_plt.loc[
    (df_plt["officialPosition"] == "DE") & (df_plt["snap_los_diff"] < 0)
].sort_values("snap_los_diff")